# model-train-eval-toggle-around-sample — ex1: eval→no_grad→sample→train: clean sampling inside a training loop

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `model-train-eval-toggle-around-sample`. Running the final beacon cell reports progress against the `GAN: model.train/eval toggle around sample` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: model.train/eval toggle around sample` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`model-train-eval-toggle-around-sample`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "model-train-eval-toggle-around-sample"
DD_SUBTOPIC = "GAN: model.train/eval toggle around sample"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## model.train() / eval() toggle around sample — quick refresher

When you sample from a generator (or any net with BatchNorm / Dropout) for LOGGING or VISUALIZATION inside a training loop, you must:

```python
model.eval()                              # turn BN to running stats, disable dropout
with t.no_grad():                         # don't track this in autograd
    samples = model(noise)
model.train()                             # restore training mode
```

**Why both `eval()` and `no_grad()`.** They do different things. `eval()` switches the BEHAVIOR of BatchNorm (use running stats, don't update them) and Dropout (no zeroing). `no_grad()` switches the GRADIENT MACHINERY (no graph, no `requires_grad` propagation). You need both for a clean sample.

**Without `eval()`.** BatchNorm computes mean/var on the noise batch (which is unrelated to your real-data running stats) AND updates those running stats — corrupting the model's BN parameters with noise. Visible later as quality regression when training resumes.

**Why restore with `model.train()`.** The next training step needs the model in training mode. Forgetting this is one of the classic GAN bugs — BatchNorm running stats freeze, gradients flow weirdly, your loss curve mysteriously plateaus.

### Exercise 1 — eval→no_grad→sample→train: clean sampling inside a training loop

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `eval() → no_grad() → forward → train()` toggle pattern to sample from a generator inside a training loop without corrupting BatchNorm running statistics.
> Keywords: gan, eval-mode, no_grad, batchnorm-running-stats
> ```

**KCs targeted:** `model-eval-no-grad-sample`, `restore-train-mode`

Implement `ex1_sample_clean(model, noise)`. The standard BN-safe sampling block:

1. Switch the model to evaluation mode: `model.eval()`.
2. Enter a `torch.no_grad()` context. INSIDE the context, run the forward pass: `samples = model(noise)`.
3. Switch the model BACK to training mode: `model.train()`.
4. Return `samples`.

Critical invariants the test checks:
- `samples.requires_grad` must be `False` (because of `no_grad`).
- `model.training` must be `True` AFTER the call (we restored it).
- The model's `BatchNorm.running_mean` and `running_var` must be UNCHANGED by the sampling call (because `eval()` makes BN use running stats and skip the update).

Input: `model` — `nn.Module` with BatchNorm layers somewhere; `noise` — input tensor of the right shape.
Output: model output tensor, with `requires_grad=False`, the model left in `train()` mode, BN running stats untouched.

The visualization plots BatchNorm's running_mean across the layer BEFORE and AFTER a sample call to confirm it didn't drift.

In [ ]:
def ex1_sample_clean(model: nn.Module, noise: Tensor) -> Tensor:
    """Sample without corrupting BN running stats: eval → no_grad → forward → train."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    # Build a small DCGAN-ish generator with BN.
    model = nn.Sequential(
        nn.ConvTranspose2d(10, 32, 4, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1, bias=False),
        nn.Tanh(),
    )
    # Prime BN running stats with a few real-data forward passes in train mode.
    model.train()
    for _ in range(3):
        _ = model(t.randn(8, 10, 1, 1))
    bn_layer = model[1]
    rmean_before = bn_layer.running_mean.detach().clone()
    rvar_before = bn_layer.running_var.detach().clone()

    # Now sample with a wildly different distribution → would shift BN stats if eval() were missing.
    weird_noise = 10.0 * t.randn(8, 10, 1, 1)
    out = ex1_sample_clean(model, weird_noise)

    # Shape sanity — output should be image-shaped.
    assert out.shape == (8, 3, 8, 8), f'unexpected output shape {tuple(out.shape)}'

    # requires_grad must be False — no_grad context.
    assert not out.requires_grad, 'samples should NOT require grad (no_grad context)'

    # Model must be left in training mode.
    assert model.training, 'model must be restored to training mode'
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            assert m.training, f'BatchNorm submodule must also be back in training mode'

    # BN running stats must NOT have changed during sampling.
    assert t.allclose(bn_layer.running_mean, rmean_before), 'BN running_mean must NOT have changed during sample'
    assert t.allclose(bn_layer.running_var, rvar_before), 'BN running_var must NOT have changed during sample'

    # Sanity — a normal forward pass (in train mode, no toggle) DOES update running stats.
    rmean_pre_train = bn_layer.running_mean.detach().clone()
    _ = model(t.randn(8, 10, 1, 1))
    assert not t.allclose(bn_layer.running_mean, rmean_pre_train), (
        'Sanity broke: plain train-mode forward should update BN running_mean. '
        'If this fires, the model has no live BN layers, not your bug.'
    )

    # Output value match — eval-mode forward + same noise gives reproducible result.
    model.train()
    rng = t.Generator().manual_seed(0)
    fixed_noise = t.randn(2, 10, 1, 1, generator=rng)
    s1 = ex1_sample_clean(model, fixed_noise)
    s2 = ex1_sample_clean(model, fixed_noise)
    assert t.allclose(s1, s2), 'two eval-mode samples on same noise must match (deterministic)'

    # --- Visualization: BN running_mean before vs after a sample call ---
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(rmean_before.numpy(), 'o-', label='before sample', color='steelblue', alpha=0.8)
    ax.plot(bn_layer.running_mean.numpy(), 'x--', label='after sample (should match)', color='coral', alpha=0.8)
    ax.set_xlabel('BN channel idx'); ax.set_ylabel('running_mean')
    ax.set_title('BN running_mean — unchanged after eval-mode sample')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_sample_clean(model: nn.Module, noise: Tensor) -> Tensor:
    model.eval()
    with t.no_grad():
        samples = model(noise)
    model.train()
    return samples
```

**`eval()` propagates to submodules.** Calling `model.eval()` recursively switches every BatchNorm and Dropout submodule. Same for `train()`. You don't have to walk the children yourself.

**Why two mechanisms (mode + no_grad).** They're orthogonal. `model.eval()` is about RUNTIME BEHAVIOR — BN uses running stats, Dropout is a no-op. `torch.no_grad()` is about AUTOGRAD — no graph is built, `requires_grad` propagation is suppressed. You need eval() for correctness; you need no_grad() for memory + speed.

**Forgetting `model.train()` is the canonical bug.** The next real training step happens with BN frozen on running stats, and with running stats that aren't being updated. The training loss subtly diverges. Always restore — even better, use a `try/finally` if you've got side effects between.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()